In [1]:
print("hi this is tic tac toe")

hi this is tic tac toe


In [2]:
from enum import Enum

class CellState(Enum):
    EMPTY = ' '
    X = 'X'
    O = 'O'
    
class currentPlayer(Enum):
    PLAYER_X = CellState.X
    PLAYER_O = CellState.O
    
class status(Enum):
    IN_PROGRESS = 0
    PLAYER_X_WINS = 1
    PLAYER_O_WINS = 2
    DRAW = 3


In [3]:
class rule_engine:
    def __init__(self):
        self.board = [[CellState.EMPTY for _ in range(3)] for _ in range(3)]
        self.current_player = currentPlayer.PLAYER_X
        self.status = status.IN_PROGRESS
        
    def get_current_player(self):
        return self.current_player
    
    def get_status(self):
        return self.status
    
    def update_status(self):
        # Check rows, columns, and diagonals for a win
        for i in range(3):
            if self.board[i][0] == self.board[i][1] == self.board[i][2] != CellState.EMPTY:
                self.status = status.PLAYER_X_WINS if self.board[i][0] == CellState.X else status.PLAYER_O_WINS
                return
            if self.board[0][i] == self.board[1][i] == self.board[2][i] != CellState.EMPTY:
                self.status = status.PLAYER_X_WINS if self.board[0][i] == CellState.X else status.PLAYER_O_WINS
                return
        if self.board[0][0] == self.board[1][1] == self.board[2][2] != CellState.EMPTY:
            self.status = status.PLAYER_X_WINS if self.board[0][0] == CellState.X else status.PLAYER_O_WINS
            return
        if self.board[0][2] == self.board[1][1] == self.board[2][0] != CellState.EMPTY:
            self.status = status.PLAYER_X_WINS if self.board[0][2] == CellState.X else status.PLAYER_O_WINS
            return
        
        # Check for draw
        if all(cell != CellState.EMPTY for row in self.board for cell in row):
            self.status = status.DRAW
            
    def get_legal_actions(self):
        legal_actions = []
        for i in range(3):
            for j in range(3):
                if self.board[i][j] == CellState.EMPTY:
                    legal_actions.append((i, j))
        return legal_actions
    
    def apply_action(self, action):
        if self.status != status.IN_PROGRESS:
            raise Exception("Game is already over.")
        if action not in self.get_legal_actions():
            raise Exception("Invalid action.")
        i, j = action
        self.board[i][j] = self.current_player.value
        self.update_status()
        self.current_player = currentPlayer.PLAYER_O if self.current_player == currentPlayer.PLAYER_X else currentPlayer.PLAYER_X
    
    def is_game_over(self):
        return self.status != status.IN_PROGRESS

In [4]:
def print_board(board):
    for row in board:
        print(' | '.join(cell.value for cell in row))
        print('-' * 5)

In [5]:
import copy

def human_agent(engine) -> tuple:
    legal_actions = engine.get_legal_actions()
    if not legal_actions:
        return None
    print("Legal actions:", legal_actions)
    print_board(engine.board)
    while True:
        try:
            action = input("Enter your move as 'row,col': ")
            i, j = map(int, action.split(','))
            if (i, j) in legal_actions:
                return (i, j)
            else:
                print("Invalid move. Try again.")
        except Exception as e:
            print(f"Error: {e}. Please enter a valid move.")

def random_agent(engine) -> tuple:
    import random
    legal_actions = engine.get_legal_actions()
    if legal_actions:
        return random.choice(legal_actions)
    return None

def minmax_agent(engine, maximizing_player) -> tuple:
    if engine.is_game_over():
        if engine.get_status() == status.PLAYER_X_WINS:
            return (1, None)
        elif engine.get_status() == status.PLAYER_O_WINS:
            return (-1, None)
        else:
            return (0, None)

    if maximizing_player:
        max_eval = float('-inf')
        best_action = None
        for action in engine.get_legal_actions():
            new_engine = copy.deepcopy(engine)
            new_engine.apply_action(action)
            eval_score, _ = minmax_agent(new_engine, False)
            if eval_score > max_eval:
                max_eval = eval_score
                best_action = action
        return (max_eval, best_action)
    else:
        min_eval = float('inf')
        best_action = None
        for action in engine.get_legal_actions():
            new_engine = copy.deepcopy(engine)
            new_engine.apply_action(action)
            eval_score, _ = minmax_agent(new_engine, True)
            if eval_score < min_eval:
                min_eval = eval_score
                best_action = action
        return (min_eval, best_action)
      


In [ ]:
'''
MATCH
 │
 ├── winner
 ├── moves
 ├── duration
 │
 └── DECISIONS
       │
       ├── Decision 1
       │     ├── action
       │     ├── duration
       │     └── search
       │
       ├── Decision 2
       │     ├── action
       │     ├── duration
       │     └── search
       │
       └── Decision N
       
MatchMetrics
├── winner: PLAYER_X_WINS
├── no_of_moves: 7
├── duration_ms: 4912.7
└── decisions
      ├── DecisionMetrics
      ├── DecisionMetrics
      ├── DecisionMetrics
      └── ...


MatchMetrics
├── winner: PLAYER_X_WINS
├── no_of_moves: 7
├── duration_ms: 4912.7
└── decisions
      ├── DecisionMetrics
      ├── DecisionMetrics
      ├── DecisionMetrics
      └── ...
'''
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class SearchMetrics:
    nodes_explored: int = 0
    terminal_nodes: int = 0
    deep_copies: int = 0
    max_depth: int = 0


@dataclass
class DecisionMetrics:
    duration_ms: float = 0.0
    search_metrics: SearchMetrics = field(default_factory=SearchMetrics)
    chosen_action: Optional[tuple] = None


@dataclass
class MatchMetrics:
    winner: status = status.IN_PROGRESS
    no_of_moves: int = 0
    duration_ms: float = 0.0
    decisions: list[DecisionMetrics] = field(default_factory=list)

In [16]:
# game loop , minmax agent vs random agent
x_count = 0
o_count = 0
draw_count = 0

for _ in range(1):  # Play 1 game
    engine = rule_engine()
    while not engine.is_game_over():
        # print_board(engine.board)
        if engine.get_current_player() == currentPlayer.PLAYER_X:
            _,action = minmax_agent(engine, True)
        else:
            action = random_agent(engine)
        if action:
            engine.apply_action(action)

    if engine.get_status() == status.PLAYER_X_WINS:
        x_count += 1
    elif engine.get_status() == status.PLAYER_O_WINS:
        o_count += 1
    else:
        draw_count += 1


print(f"Player X wins(minmax): {x_count}")
print(f"Player O wins(random): {o_count}")
print(f"Draws: {draw_count}")

Player X wins(minmax): 1
Player O wins(random): 0
Draws: 0


In [19]:
x_count=0
y_count=0
draw_count=0

for _ in range(5):  # Play 5 games
    engine = rule_engine()
    while not engine.is_game_over():
        # print_board(engine.board)
        if engine.get_current_player() == currentPlayer.PLAYER_X:
            action = random_agent(engine)
        else:
            _,action = minmax_agent(engine, False)
        if action:
            engine.apply_action(action)

    if engine.get_status() == status.PLAYER_X_WINS:
        x_count += 1
    elif engine.get_status() == status.PLAYER_O_WINS:
        y_count += 1
    else:
        draw_count += 1

print(f"Player X wins(random): {x_count}")
print(f"Player O wins(minmax): {y_count}")
print(f"Draws: {draw_count}")

Player X wins(random): 0
Player O wins(minmax): 4
Draws: 1


In [20]:
x_count=0
y_count=0
draw_count=0

for _ in range(1):  # Play 1 game
    engine = rule_engine()
    while not engine.is_game_over():
        # print_board(engine.board)
        if engine.get_current_player() == currentPlayer.PLAYER_X:
            _,action = minmax_agent(engine, True)
        else:
            _,action = minmax_agent(engine, False)
        if action:
            engine.apply_action(action)

    print(f"Result: {engine.get_status()}")
    if engine.get_status() == status.PLAYER_X_WINS:
        x_count += 1
    elif engine.get_status() == status.PLAYER_O_WINS:
        y_count += 1
    else:
        draw_count += 1

print(f"Player X wins(minmax): {x_count}")
print(f"Player O wins(minmax): {y_count}")
print(f"Draws: {draw_count}")

Result: status.DRAW
Player X wins(minmax): 0
Player O wins(minmax): 0
Draws: 1


In [9]:
x_count=0
y_count=0
draw_count=0

engine = rule_engine()
while not engine.is_game_over():
    if engine.get_current_player() == currentPlayer.PLAYER_X:
        action = human_agent(engine)
    else:
        _,action = minmax_agent(engine, False)
    if action:
        engine.apply_action(action)
    

Legal actions: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
  |   |  
-----
  |   |  
-----
  |   |  
-----
Legal actions: [(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1), (2, 2)]
O |   |  
-----
  | X |  
-----
  |   |  
-----
Invalid move. Try again.
Legal actions: [(0, 1), (1, 0), (1, 2), (2, 1), (2, 2)]
O |   | X
-----
  | X |  
-----
O |   |  
-----
Legal actions: [(0, 1), (2, 1), (2, 2)]
O |   | X
-----
X | X | O
-----
O |   |  
-----
Legal actions: [(2, 2)]
O | X | X
-----
X | X | O
-----
O | O |  
-----
